# Product Master Data API — manual test

Posts every file in `product_data/` to the product API, then inspects what was stored
and what the rule engine wrote.

**Before running:**
1. `python product_seed_validation_rules.py`  (one-time, loads the example product rules)
2. `python product_run.py`  (starts the API on http://localhost:8001)
3. Optional: run the reset cell at the bottom for a clean slate.

Every product that fails a rule gets one log file in `product_validation_logs/`
(section 6), naming the people registered for its `product_code` (section 1).

Re-posting is safe: each product **replaces** its row in `product_latest` (and its
detail rows), while `product_history` and `product_validation_results` keep every posting.

Bulk alternative without the server: `python product_load_data.py`.

In [ ]:
import os
import json
import sqlite3

import requests
import pandas as pd

BASE_URL = "http://localhost:8001"

# resolve the project root (folder containing product_run.py) so relative
# paths work no matter where the kernel was started
_here = os.getcwd()
while not os.path.exists(os.path.join(_here, "product_run.py")) and os.path.dirname(_here) != _here:
    _here = os.path.dirname(_here)
PROJECT_ROOT = _here

DATA_DIR = os.path.join(PROJECT_ROOT, "product_data")
DATA_FILES = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".json"))
# failed-validation logs: same default the app uses (product_api/product_config.py)
LOG_DIR = os.path.join(PROJECT_ROOT, os.environ.get("PRODUCT_LOG_DIR", "product_validation_logs"))

print("project root:", PROJECT_ROOT)
print("data files  :", DATA_FILES)
print("liveness    :", requests.get(f"{BASE_URL}/docs").status_code)

## 1. Register who is responsible for each product

A contact is linked to a **`product_code`** — the last part of `ProductDefinitionKey`
(`urn:inexto:id:prdd:tobacco:pmi:FA003115` → `FA003115`), shared by every level of
that product. Register contacts before posting, so the log files can name them.
Re-running this cell is safe: identical rows come back as `skipped`.

In [ ]:
contacts = [
    {"product_code": "FA003115", "person_name": "Aisyah Rahman", "contact": "aisyah.rahman@example.com"},
    {"product_code": "FA003115", "person_name": "L&M Brand Team", "contact": "Teams: L&M Brand Team"},
    {"product_code": "DK001129", "person_name": "Budi Santoso", "contact": "budi.santoso@example.com"},
]

resp = requests.post(f"{BASE_URL}/product-responsible-contacts", json=contacts)
resp.raise_for_status()
print("inserted:", resp.json()["inserted"], "| skipped:", resp.json()["skipped"])

# matching_products = products in product_latest with that code (0 until they are posted)
pd.DataFrame(requests.get(f"{BASE_URL}/product-responsible-contacts").json())

## 2. POST every product file

In [ ]:
def upload_product_file(path, base_url=BASE_URL):
    """POST one product JSON file to /products/upload-file; return the summary."""
    with open(path, "rb") as f:
        resp = requests.post(
            f"{base_url}/products/upload-file",
            files={"file": (os.path.basename(path), f, "application/json")},
        )
    if resp.status_code >= 400:
        raise RuntimeError(f"{resp.status_code}: {resp.text}")
    return resp.json()


posted = []
for name in DATA_FILES:
    summary = upload_product_file(os.path.join(DATA_DIR, name))
    print(f"{name}: {summary['total']} products, {summary['new_records']} new, "
          f"{summary['updated_records']} updated, {summary['failed_validation']} failed validation")
    posted.extend(summary["details"])

pd.DataFrame(posted)

## 3. GET /products  (one row per packaging level)

In [ ]:
products = pd.DataFrame(requests.get(f"{BASE_URL}/products").json())
products[["id", "primary_product_key", "parent_product_key", "level", "quantity",
          "product_code", "product_type", "description", "product_category", "brand_family"]]

## 4. Validation results

In [ ]:
summary = pd.DataFrame(requests.get(f"{BASE_URL}/products/validation-results/summary").json())
summary

In [ ]:
# pass/fail matrix: one row per product, one column per rule
latest = pd.DataFrame(requests.get(f"{BASE_URL}/products/validation-results/latest").json())
if not latest.empty:
    matrix = latest.pivot_table(
        index=["primary_product_key", "product_type"],
        columns="rule_id",
        values="passed",
        aggfunc="last",
    )
    display(matrix.replace({1: "PASS", 0: "FAIL", True: "PASS", False: "FAIL"}))

    print()
    print("Failures only:")
    display(latest[latest["passed"].isin([0, False])][
        ["primary_product_key", "product_type", "rule_id", "severity", "status", "description"]
    ])

## 5. Lookup by any key (e.g. a GTIN)

`rule_fields` is exactly what the rule engine sees for a product — copy field names
from there when writing a rule.

In [ ]:
KEY = "urn:inexto:id:prd:gtin:05410706508796"   # the L&M shipping case in product.json

resp = requests.get(f"{BASE_URL}/products/lookup", params={"key": KEY})
print(resp.status_code)
for match in resp.json():
    print("matched on:", match["matched_on"], "->", match["product"]["primary_product_key"])
    display(pd.DataFrame(match["alternate_keys"]))
    display(pd.DataFrame(match["children"]))
    display(pd.DataFrame(match["responsible_contacts"]))
    display(pd.Series(match["rule_fields"], name="value").to_frame())

## 6. Failed-validation log files

One JSON file per product that failed at least one rule, named
`log_<first 12 characters of product_hash>_<YYYYMMDD>.json`. All the rules it failed are
in that one file; posting the product again on the same day adds a new entry to its
`validations` list. The POST response names each product's file in `log_file`.

In [ ]:
paths = []
if os.path.isdir(LOG_DIR):
    paths = [os.path.join(LOG_DIR, name) for name in os.listdir(LOG_DIR) if name.endswith(".json")]
paths.sort(key=os.path.getmtime)

print(f"{len(paths)} log file(s) in {LOG_DIR}")
for path in paths:
    print("  ", os.path.basename(path))

if paths:
    print()
    print("newest:", os.path.basename(paths[-1]))
    with open(paths[-1], encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

## Reset (optional) — clears the product data tables and the log files

The rule tables (`product_val_*`) are left untouched, so no re-seed is needed.
The API server can keep running.

In [ ]:
DB_PATH = os.path.join(PROJECT_ROOT, os.environ.get("PRODUCT_DB_NAME", "product_database.db"))
DATA_TABLES = ["product_history", "product_latest", "product_alternate_key", "product_unit",
               "product_property", "product_validation_results",
               "product_latest_validation_rules", "product_validation_summary",
               "product_responsible_contact"]


def reset_product_data():
    if not os.path.exists(DB_PATH):
        print("no database file yet - nothing to reset.")
        return
    with sqlite3.connect(DB_PATH) as conn:
        for table in DATA_TABLES:
            conn.execute(f"DELETE FROM {table}")
        # restart AUTOINCREMENT ids at 1
        conn.execute(
            f"DELETE FROM sqlite_sequence WHERE name IN ({','.join('?' * len(DATA_TABLES))})",
            DATA_TABLES,
        )
    print("cleared:", ", ".join(DATA_TABLES))


def reset_product_logs():
    names = sorted(os.listdir(LOG_DIR)) if os.path.isdir(LOG_DIR) else []
    names = [name for name in names if name.startswith("log_")]
    for name in names:
        os.remove(os.path.join(LOG_DIR, name))
    print(f"deleted {len(names)} log file(s) from {LOG_DIR}")


# reset_product_data()   # uncomment to run
# reset_product_logs()   # uncomment to run